# 基于 Criteo 的广告点击率预估

比较类别线性权重与 FM 二阶交互对点击概率预测的贡献。请在新的 Python 内核中从上到下运行。需要 pandas、NumPy、requests、scikit-learn、PyTorch 和联网访问数据源。

## 数据与复现
CriteoClickLogs 固定版本 a1a472783e135c6786617567f51bb64bfc4b031a，按原顺序取 day_0.gz 前 100,000 行。40 列为 label、I1–I13、C1–C26，共 2,684 次点击。早期 200 条练习数据与正式结果分开。
两次 train_test_split 都按当次输入的 label 分层，random_state=42。训练 70,000 条/1,879 次点击，验证 15,000 条/403 次点击，测试 15,000 条/402 次点击。测试集留到方案锁定后评估。


In [ ]:
import gzip
from itertools import islice
from pathlib import Path

import requests

revision = "a1a472783e135c6786617567f51bb64bfc4b031a"
url = f"https://huggingface.co/datasets/criteo/CriteoClickLogs/resolve/{revision}/day_0.gz"
target = Path.cwd() / "criteo_day0_first100k.tsv"
partial = target.with_suffix(".part")
limit = 100_000
rows_read = 0

with requests.get(url, stream=True, timeout=(15, 60)) as response:
    response.raise_for_status()
    with gzip.GzipFile(fileobj=response.raw) as source, partial.open("wb") as out:
        for rows_read, line in enumerate(islice(source, limit), start=1):
            fields = line.rstrip(b"\r\n").split(b"\t")
            if len(fields) != 40:
                raise ValueError(f"第 {rows_read} 行有 {len(fields)} 个字段，预期 40 个")
            out.write(line)

if rows_read != limit:
    raise ValueError(f"只读到 {rows_read} 行，目标是 {limit} 行")

partial.replace(target)
print("来源：", url)
print("本地文件：", target)
print("已逐行核验字段数的曝光条数：", rows_read)

In [41]:
import pandas as pd

columns = (
    ["label"]
    + [f"I{i}" for i in range(1, 14)]
    + [f"C{i}" for i in range(1, 27)]
)

df = pd.read_csv(
    target,
    sep="\t",
    header=None,
    names=columns,
    dtype={name: "string" for name in columns if name.startswith("C")},
)

display(df.iloc[:3, [0, 1, 13, 14, 39]])

,label,I1,I13,C1,C26
0,1,5.0,NaN,62770d79,ed10571d
1,0,32.0,5.0,e5f3fd8d,b757e957
2,0,NaN,1.0,62770d79,ed10571d


In [42]:
print(df.shape)
print(df["label"].value_counts(dropna=False))
exposures = len(df)
print("曝光数",exposures)
clicks = df["label"].sum()
print("点击数",clicks)
CTR = clicks/exposures
print("样本CTR",CTR)

(100000, 40)
label
0    97316
1     2684
Name: count, dtype: int64
曝光数 100000
点击数 2684
样本CTR 0.02684


In [43]:
from sklearn.model_selection import train_test_split

seed = 42
assert df.shape == (100_000,40)
assert df["label"].sum() == 2_684

train_df,remaining_df = train_test_split(
    df,
    test_size=0.3,
    random_state=seed,
    stratify=df["label"]
)
val_df,test_df = train_test_split(
    remaining_df,
    test_size=0.5,
    random_state=seed,
    stratify=remaining_df["label"]
)
print("3个集合的数据依次为：")
exposures_total = 0
clicks_total = 0
for subset in [train_df,val_df,test_df]:
  exposures = len(subset)
  clicks = subset["label"].sum()
  CTR = clicks/exposures
  exposures_total+=exposures
  clicks_total+=clicks
  print(f"曝光量:{exposures},点击量:{clicks},CTR:{CTR}")

assert exposures_total == len(df)
assert clicks_total == df["label"].sum()
print("划分后原表形状：",df.shape)
print(exposures_total)
print(clicks_total)

3个集合的数据依次为：
曝光量:70000,点击量:1879,CTR:0.026842857142857144
曝光量:15000,点击量:403,CTR:0.026866666666666667
曝光量:15000,点击量:402,CTR:0.0268
划分后原表形状： (100000, 40)
100000
2684


In [44]:
assert train_df.index.intersection(val_df.index).empty
assert train_df.index.intersection(test_df.index).empty
assert val_df.index.intersection(test_df.index).empty
print("三个集合两两无重叠")

三个集合两两无重叠


## 常数基线与数值预处理

In [45]:
import numpy as np
from sklearn.metrics import roc_auc_score,log_loss

p_train = train_df["label"].sum()/len(train_df)
y_val = val_df["label"]
baseline_pred = np.full(len(val_df),p_train)

baseline_auc = roc_auc_score(y_val,baseline_pred)
baseline_loss = log_loss(y_val,baseline_pred)

print("训练集点击率：", p_train)
print("预测数组形状：", baseline_pred.shape)
print("前 5 个预测概率：", baseline_pred[:5])
print(f"验证集 AUC = {baseline_auc:.6f}")
print(f"验证集 Log Loss = {baseline_loss:.6f}")


训练集点击率： 0.026842857142857144
预测数组形状： (15000,)
前 5 个预测概率： [0.02684286 0.02684286 0.02684286 0.02684286 0.02684286]
验证集 AUC = 0.500000
验证集 Log Loss = 0.123676


In [46]:
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

num_cols = [f"I{i}" for i in range(1,14)]

X_train_num = train_df[num_cols]
X_val_num = val_df[num_cols]

num_preprocessor = make_pipeline(
    SimpleImputer(strategy="median"),
    StandardScaler()
)

X_train_scaled = num_preprocessor.fit_transform(X_train_num)
X_val_scaled = num_preprocessor.transform(X_val_num)

print("训练集形状：", X_train_scaled.shape)
print("验证集形状：", X_val_scaled.shape)
print("处理前，训练集缺失单元格数：", X_train_num.isna().sum().sum())
print("处理后，训练集缺失单元格数：", np.isnan(X_train_scaled).sum())
print("处理后，验证集缺失单元格数：", np.isnan(X_val_scaled).sum())

训练集形状： (70000, 13)
验证集形状： (15000, 13)
处理前，训练集缺失单元格数： 171103
处理后，训练集缺失单元格数： 0
处理后，验证集缺失单元格数： 0


In [47]:
print("预处理器保存的 I1 填补值：",
      num_preprocessor.named_steps["simpleimputer"].statistics_[0])
print("训练集 I1 中位数：", X_train_num["I1"].median())

预处理器保存的 I1 填补值： 7.0
训练集 I1 中位数： 7.0


## A · 数值逻辑回归

先用一个批次演示参数更新，正式训练重新初始化模型。

In [48]:
import numpy as np
import torch
from torch import nn

xb = torch.tensor(X_train_scaled[:512], dtype=torch.float32)
yb = torch.tensor(train_df["label"].iloc[:512].to_numpy(), dtype=torch.float32)

model_a = nn.Linear(13, 1)
with torch.no_grad():
    model_a.weight.zero_()
    model_a.bias.fill_(float(np.log(p_train / (1 - p_train))))

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(model_a.parameters(), lr=0.01)

optimizer.zero_grad()
logits = model_a(xb).squeeze(-1)
print("更新前，前 3 条点击概率：", torch.sigmoid(logits[:3]).detach().numpy())

loss_before = criterion(logits,yb)
loss_before.backward()  # 计算梯度
optimizer.step()  # 用梯度更新参数

with torch.no_grad():
    loss_after = criterion(model_a(xb).squeeze(-1), yb)

print(f"本批更新前损失：{loss_before.item():.8f}")
print(f"本批更新后损失：{loss_after.item():.8f}")

更新前，前 3 条点击概率： [0.02684285 0.02684285 0.02684285]
本批更新前损失：0.12538874
本批更新后损失：0.12538427


In [49]:
from torch.utils.data import DataLoader,TensorDataset

X_train_t = torch.tensor(X_train_scaled,dtype=torch.float32)
y_train_t = torch.tensor(train_df["label"].to_numpy(),dtype=torch.float32)
X_val_t = torch.tensor(X_val_scaled,dtype=torch.float32)

train_loader = DataLoader(
    TensorDataset(X_train_t,y_train_t),
    batch_size=512,
    shuffle=True,
    generator=torch.Generator().manual_seed(42),
)
model_a = nn.Linear(13,1)
with torch.no_grad():
  model_a.weight.zero_()
  model_a.bias.fill_(float(np.log(p_train/(1-p_train))))
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.SGD(model_a.parameters(),lr=0.01)

model_a.train()
for epoch in range(5):
  total_loss = 0.0
  for xb,yb in train_loader:
    optimizer.zero_grad()
    logits = model_a(xb).squeeze(-1)
    loss = criterion(logits,yb)
    loss.backward()
    optimizer.step()
    total_loss += loss.item()*len(yb)
  print(f"第{epoch+1}轮训练平均损失：{total_loss/len(y_train_t):.6f}")

第1轮训练平均损失：0.123370
第2轮训练平均损失：0.122962
第3轮训练平均损失：0.122615
第4轮训练平均损失：0.122332
第5轮训练平均损失：0.122104


In [50]:
model_a.eval()
with torch.no_grad():
  val_logits = model_a(X_val_t).squeeze(-1)
  val_prob_a = torch.sigmoid(val_logits).numpy()

auc_a = roc_auc_score(y_val,val_prob_a)
loss_a = log_loss(y_val,val_prob_a)

print(f"模型A验证集 AUC:{auc_a}")
print(f"模型A验证集 Log Loss:{loss_a:.6f}")

模型A验证集 AUC:0.6405998649234665
模型A验证集 Log Loss:0.122048


## 类别编码

每列独立建立训练词表。0 为缺失，1 为训练未见，2 为低频，3 起为常见类别。

In [51]:
import pandas as pd

min_count = 5
c1_train = train_df["C1"]
c1_val = val_df['C1']

c1_counts = c1_train.value_counts(dropna=True)
print(c1_counts.head())

C1
ad98e872    16867
788a5d5b     6160
265366bf     4716
e5f3fd8d     1744
4f50727e     1519
Name: count, dtype: Int64


In [52]:
c1_vocab = {}
for (value,count) in c1_counts.items():
  if count>=min_count:
    c1_vocab[value]=3+len(c1_vocab)
print(list(c1_vocab.items())[:3])

def encode_c1(value):
  if pd.isna(value):
    return 0
  count = c1_counts.get(value,0)

  if count == 0:
    return 1

  if count<min_count:
    return 2

  return c1_vocab[value]

print("缺失值的ID：",encode_c1(None))
print("ad98e872的ID：",encode_c1("ad98e872"))

[('ad98e872', 3), ('788a5d5b', 4), ('265366bf', 5)]
缺失值的ID： 0
ad98e872的ID： 3


In [53]:
c1_train_ids = c1_train.map(encode_c1)
c1_val_ids = c1_val.map(encode_c1)

In [54]:
for name, raw, ids in [
    ("训练集", c1_train, c1_train_ids),
    ("验证集", c1_val, c1_val_ids),
]:
    # 训练计数里查不到的值按 0 次处理，缺失值仍由 raw.isna() 单独识别
    counts = raw.map(c1_counts).fillna(0)

    missing = raw.isna()
    unseen = raw.notna() & counts.eq(0)
    rare = counts.between(1, min_count - 1)
    common = counts.ge(min_count)

    # 根据四条规则生成每一行应有的 ID
    expected = pd.Series(0, index=raw.index, dtype="Int64")
    expected.loc[unseen] = 1
    expected.loc[rare] = 2
    expected.loc[common] = raw.loc[common].map(c1_vocab).astype("Int64")

    # 缺少 ID 也算检查失败
    wrong = ids.isna() | expected.isna() | ids.ne(expected).fillna(True)
    if wrong.any():
        details = pd.DataFrame({
            "C1 原值": raw,
            "训练出现次数": counts,
            "实际 ID": ids,
            "预期 ID": expected,
        })
        print(details.loc[wrong].head())
        raise AssertionError(f"{name}有 {wrong.sum()} 条 C1 编码不一致")

    print(name, "缺失", missing.sum(), "未见", unseen.sum(),
          "低频", rare.sum(), "常见", common.sum())

print("C1 编码检查通过")

训练集 缺失 2030 未见 0 低频 19959 常见 48011
验证集 缺失 424 未见 3198 低频 1216 常见 10162
C1 编码检查通过


In [55]:
cat_cols = [f"C{i}" for i in range(1,27)]
counts_by_col = {}
vocabs_by_col = {}

for col in cat_cols:
  counts = train_df[col].value_counts(dropna=True)
  vocab = {}

  for value,count in counts.items():
    if count >= min_count:
      vocab[value] = 3+len(vocab)
  counts_by_col[col] = counts
  vocabs_by_col[col] = vocab

assert counts_by_col["C1"].equals(c1_counts)
assert vocabs_by_col["C1"] == c1_vocab

print("已建词表的字段数：",len(vocabs_by_col))
for col in ["C1","C2"]:
  print(col,"有独立ID的类别数：",len(vocabs_by_col[col]))

已建词表的字段数： 26
C1 有独立ID的类别数： 632
C2 有独立ID的类别数： 1281


In [56]:
def encode_column(raw,col):
  counts = counts_by_col[col]
  vocab = vocabs_by_col[col]#是个字典

  train_count = raw.map(counts)
  ids = raw.map(vocab)

  ids.loc[raw.isna()] = 0
  ids.loc[raw.notna() & train_count.isna()] = 1
  ids.loc[train_count.between(1,min_count-1)] = 2
  return ids.astype("int64")

train_cat_ids = pd.DataFrame(index=train_df.index)
val_cat_ids = pd.DataFrame(index=val_df.index)

for col in cat_cols:
  train_cat_ids[col] = encode_column(train_df[col],col)
  val_cat_ids[col] = encode_column(val_df[col],col)
# 用第 1 节逐个值编码的结果，核对现在整列编码的结果
assert train_cat_ids["C1"].equals(c1_train_ids)
assert val_cat_ids["C1"].equals(c1_val_ids)
assert not train_cat_ids.eq(1).any().any()

print("训练类别表形状：", train_cat_ids.shape)
print("验证类别表形状：", val_cat_ids.shape)
print("验证集前 3 条曝光的前 4 个类别 ID：")
print(val_cat_ids.iloc[:3, :4])


训练类别表形状： (70000, 26)
验证类别表形状： (15000, 26)
验证集前 3 条曝光的前 4 个类别 ID：
       C1   C2  C3   C4
20017   4   13  12    2
8093    1  281  34   93
30791   5   54  58  207


## B · 数值与类别线性权重

In [57]:
import torch
from torch import nn

c1_num_ids = len(vocabs_by_col["C1"])+3
c1_weights = nn.Embedding(c1_num_ids,1)
assert c1_weights.num_embeddings == max(vocabs_by_col["C1"].values()) + 1
nn.init.zeros_(c1_weights.weight)

# 取你刚才打印的三条曝光，按 C1 的整数 ID 查表
c1_ids = torch.tensor(val_cat_ids["C1"].iloc[:3].to_numpy(), dtype=torch.long)
c1_contribution = c1_weights(c1_ids).squeeze(-1)

print("C1 权重表形状：", tuple(c1_weights.weight.shape))
print("查表使用的 C1 ID：", c1_ids.tolist())
print("每条曝光的 C1 贡献：", c1_contribution.tolist())

C1 权重表形状： (635, 1)
查表使用的 C1 ID： [4, 1, 5]
每条曝光的 C1 贡献： [0.0, 0.0, 0.0]


In [58]:
cat_tables = nn.ModuleDict()

for col in cat_cols:
  num_ids = len(vocabs_by_col[col])+3
  table = nn.Embedding(num_ids,1)
  nn.init.zeros_(table.weight)
  cat_tables[col] = table

# 每个字段取出前三条曝光对应的三个权重
field_contributions = []
for col in cat_cols:
    ids = torch.tensor(val_cat_ids[col].iloc[:3].to_numpy(), dtype=torch.long)
    weights = cat_tables[col](ids).squeeze(-1)
    field_contributions.append(weights)

# 每行是一条曝光，每列是一个字段贡献的权重
by_field = torch.stack(field_contributions, dim=1)
cat_total = by_field.sum(dim=1)

assert by_field.shape == (3, len(cat_cols))
assert cat_total.shape == (3,)

print("C2 权重表形状：", tuple(cat_tables["C2"].weight.shape))
print("逐字段贡献形状：", tuple(by_field.shape))
print("三条曝光各自的类别总贡献：", cat_total.tolist())


C2 权重表形状： (1284, 1)
逐字段贡献形状： (3, 26)
三条曝光各自的类别总贡献： [0.0, 0.0, 0.0]


In [59]:
import math

class ModelB(nn.Module):
    def __init__(self, train_ctr, cat_cols, vocabs_by_col):
        super().__init__()
        self.numeric = nn.Linear(13, 1)
        nn.init.zeros_(self.numeric.weight)
        nn.init.constant_(self.numeric.bias, math.log(train_ctr / (1 - train_ctr)))

        self.cat_tables = nn.ModuleDict()
        for col in cat_cols:
            table = nn.Embedding(len(vocabs_by_col[col]) + 3, 1)
            nn.init.zeros_(table.weight)
            self.cat_tables[col] = table

    def forward(self, x_num, x_cat):
        num_logit = self.numeric(x_num).squeeze(-1)

        contributions = []
        for j, table in enumerate(self.cat_tables.values()):
            weights = table(x_cat[:, j]).squeeze(-1)
            contributions.append(weights)

        cat_total = torch.stack(contributions, dim=1).sum(dim=1)
        return num_logit+cat_total


train_ctr = float(train_df["label"].mean())
model_b = ModelB(train_ctr, cat_cols, vocabs_by_col)

# 取同三条曝光的数值特征和类别 ID，检查初始预测
x_num = torch.as_tensor(X_val_scaled[:3], dtype=torch.float32)
x_cat = torch.as_tensor(val_cat_ids[cat_cols].iloc[:3].to_numpy(), dtype=torch.long)

with torch.no_grad():
    logits = model_b(x_num, x_cat)
    probs = torch.sigmoid(logits)

assert logits.shape == (3,)
assert torch.allclose(probs, torch.full_like(probs, train_ctr))
print("初始 logits：", logits.tolist())
print("初始点击概率：", probs.tolist())
print("训练集 CTR：", train_ctr)

初始 logits： [-3.590545892715454, -3.590545892715454, -3.590545892715454]
初始点击概率： [0.026842854917049408, 0.026842854917049408, 0.026842854917049408]
训练集 CTR： 0.026842857142857144


In [60]:
from torch.utils.data import DataLoader, TensorDataset

assert train_cat_ids.index.equals(train_df.index)

train_data_b = TensorDataset(
    torch.as_tensor(X_train_scaled, dtype=torch.float32),
    torch.as_tensor(train_cat_ids[cat_cols].to_numpy(), dtype=torch.long),
    torch.as_tensor(train_df["label"].to_numpy(), dtype=torch.float32),
)

# 每次重跑本单元格，都从相同的初始参数和打乱种子开始
model_b = ModelB(train_ctr, cat_cols, vocabs_by_col)
loader_b = DataLoader(
    train_data_b,
    batch_size=512,
    shuffle=True,
    generator=torch.Generator().manual_seed(42),
)
criterion_b = nn.BCEWithLogitsLoss()
optimizer_b = torch.optim.SGD(model_b.parameters(), lr=0.01)

model_b.train()
for epoch in range(5):
    loss_sum = 0.0
    for x_num_batch, x_cat_batch, y_batch in loader_b:
        optimizer_b.zero_grad()
        logits = model_b(x_num_batch,x_cat_batch)
        loss = criterion_b(logits,y_batch)
        loss.backward()
        optimizer_b.step()
        loss_sum += loss.item() * len(y_batch)

    print(f"第 {epoch + 1} 轮训练平均损失：{loss_sum / len(train_data_b):.6f}")

第 1 轮训练平均损失：0.123285
第 2 轮训练平均损失：0.122704
第 3 轮训练平均损失：0.122205
第 4 轮训练平均损失：0.121784
第 5 轮训练平均损失：0.121432


In [61]:
from sklearn.metrics import roc_auc_score, log_loss

assert val_cat_ids.index.equals(val_df.index)
x_num_val = torch.as_tensor(X_val_scaled, dtype=torch.float32)
x_cat_val = torch.as_tensor(val_cat_ids[cat_cols].to_numpy(), dtype=torch.long)

model_b.eval()
with torch.no_grad():
    val_logits_b = model_b(x_num_val, x_cat_val)
    val_prob_b = torch.sigmoid(val_logits_b).cpu().numpy()

assert val_prob_b.shape == (len(y_val),)

auc_b = roc_auc_score(y_val,val_prob_b)
loss_b = log_loss(y_val,val_prob_b)

# 用原始预测重算 A 的指标，避免用四舍五入后的数字求差
auc_a = roc_auc_score(y_val, val_prob_a)
loss_a = log_loss(y_val, val_prob_a)

print(f"模型 A：AUC={auc_a:.6f}，Log Loss={loss_a:.6f}")
print(f"模型 B：AUC={auc_b:.6f}，Log Loss={loss_b:.6f}")
print(f"B 减 A：ΔAUC={auc_b - auc_a:+.6f}，ΔLog Loss={loss_b - loss_a:+.6f}")

模型 A：AUC=0.640600，Log Loss=0.122048
模型 B：AUC=0.681304，Log Loss=0.121419
B 减 A：ΔAUC=+0.040704，ΔLog Loss=-0.000629


## C · FM 二阶交互

先用三个向量核对交互公式，再训练 39 个字段的模型。

In [62]:
v_c1 = torch.tensor([0.2, 0.4])
v_c2_a = torch.tensor([0.5, 0.1])
v_c2_b = torch.tensor([-0.5, 0.1])

def pair_interaction(v_left, v_right):
    return sum(v_left*v_right)

print("C1 与 C2 甲的交互贡献：", round(pair_interaction(v_c1, v_c2_a).item(), 4))
print("C1 与 C2 乙的交互贡献：", round(pair_interaction(v_c1, v_c2_b).item(), 4))

C1 与 C2 甲的交互贡献： 0.14
C1 与 C2 乙的交互贡献： -0.06


In [63]:
# 教学示例：同一条曝光的三个字段，各取出一个二维向量
vectors = torch.stack([v_c1, v_c2_a, torch.tensor([0.3, -0.2])])
n_fields = len(vectors)
pair_scores = []

for i in range(n_fields):
    for j in range(i+1, n_fields):
        score = pair_interaction(vectors[i], vectors[j])
        pair_scores.append(score)
        print(f"C{i + 1} 和 C{j + 1}：{score.item():.4f}")

assert len(pair_scores) == 3
interaction_total = torch.stack(pair_scores).sum()
print(f"交互总贡献：{interaction_total.item():.4f}")

C1 和 C2：0.1400
C1 和 C3：-0.0200
C2 和 C3：0.1300
交互总贡献：0.2500


In [64]:
def fm_interaction(v):
    square_of_sum = v.sum(dim=1).square()
    sum_of_squares = v.square().sum(dim=1)
    return (square_of_sum-sum_of_squares).sum(dim=-1)/2

# 把上一节的一条曝光整理成大小为 1 的 batch
v_batch = vectors.unsqueeze(0)
fast_total = fm_interaction(v_batch)

assert fast_total.shape == (1,)
torch.testing.assert_close(fast_total[0], interaction_total)

print("输入形状：", tuple(v_batch.shape))
print("逐对计算：", interaction_total.item())
print("FM 公式：", fast_total.item())

输入形状： (1, 3, 2)
逐对计算： 0.25
FM 公式： 0.25


In [65]:
class ModelC(ModelB):
    def __init__(self, train_ctr, cat_cols, vocabs_by_col, k=8):
        super().__init__(train_ctr, cat_cols, vocabs_by_col)

        self.num_factors = nn.Parameter(torch.empty(13, k))
        nn.init.normal_(self.num_factors, mean=0.0, std=0.01)

        self.cat_factors = nn.ModuleDict()
        for col in cat_cols:
            num_ids = len(vocabs_by_col[col]) + 3
            table = nn.Embedding(num_ids, k, padding_idx=1)
            nn.init.normal_(table.weight, mean=0.0, std=0.01)
            with torch.no_grad():
                table.weight[1].zero_()
            self.cat_factors[col] = table

    def forward(self, x_num, x_cat):
        linear_logit = super().forward(x_num, x_cat)

        # [批大小, 13, 1] × [13, k] → [批大小, 13, k]
        num_vectors = x_num.unsqueeze(-1) * self.num_factors

        cat_vectors = []
        for j, table in enumerate(self.cat_factors.values()):
            cat_vectors.append(table(x_cat[:, j]))
        cat_vectors = torch.stack(cat_vectors, dim=1)

        # 沿字段维拼接，得到 [批大小, 39, k]
        all_vectors = torch.cat([num_vectors, cat_vectors], dim=1)
        return linear_logit+fm_interaction(all_vectors)


torch.manual_seed(42)
model_c = ModelC(train_ctr, cat_cols, vocabs_by_col, k=8)

with torch.no_grad():
    initial_logits_c = model_c(x_num_val[:3], x_cat_val[:3])

assert initial_logits_c.shape == (3,)
assert torch.isfinite(initial_logits_c).all()
print("C1 交互向量表形状：", tuple(model_c.cat_factors["C1"].weight.shape))
print("前三条曝光的初始 logits：", initial_logits_c.tolist())

C1 交互向量表形状： (635, 8)
前三条曝光的初始 logits： [-3.5782220363616943, -3.609647750854492, -3.5906896591186523]


In [66]:
torch.manual_seed(42)
model_c = ModelC(train_ctr, cat_cols, vocabs_by_col, k=8)

loader_c = DataLoader(
    train_data_b,
    batch_size=512,
    shuffle=True,
    generator=torch.Generator().manual_seed(42),
)
criterion_c = nn.BCEWithLogitsLoss()
optimizer_c = torch.optim.SGD(model_c.parameters(), lr=0.01)

model_c.train()
for epoch in range(5):
    loss_sum = 0.0
    for x_num_batch, x_cat_batch, y_batch in loader_c:
        optimizer_c.zero_grad()
        logits = model_c(x_num_batch, x_cat_batch)
        loss = criterion_c(logits, y_batch)
        loss.backward()
        optimizer_c.step()
        loss_sum += loss.item() * len(y_batch)

    print(f"第 {epoch + 1} 轮训练平均损失：{loss_sum / len(train_data_b):.6f}")

model_c.eval()
with torch.no_grad():
    val_logits_c = model_c(x_num_val, x_cat_val)
    val_prob_c = torch.sigmoid(val_logits_c).cpu().numpy()

assert val_prob_c.shape == val_prob_b.shape
auc_c = roc_auc_score(y_val, val_prob_c)
loss_c = log_loss(y_val, val_prob_c)

print(f"模型 B：AUC={auc_b:.6f}，Log Loss={loss_b:.6f}")
print(f"模型 C：AUC={auc_c:.6f}，Log Loss={loss_c:.6f}")
print(f"C 减 B：ΔAUC={auc_c - auc_b:+.6f}，ΔLog Loss={loss_c - loss_b:+.6f}")

第 1 轮训练平均损失：0.123305
第 2 轮训练平均损失：0.122716
第 3 轮训练平均损失：0.122214
第 4 轮训练平均损失：0.121792
第 5 轮训练平均损失：0.121436
模型 B：AUC=0.681304，Log Loss=0.121419
模型 C：AUC=0.680239，Log Loss=0.121419
C 减 B：ΔAUC=-0.001065，ΔLog Loss=-0.000001


## 验证分组与模型选择

In [67]:
import numpy as np
import pandas as pd

# ① 逐行判断：这条曝光的 26 个类别 ID 中，是否至少有一个等于 1？
has_unseen = val_cat_ids.eq(1).any(axis=1)

# 两列都按当前验证集的行顺序放入表中，避免 pandas 按原索引自动对齐
val_group_df = pd.DataFrame({
    "has_unseen": has_unseen.to_numpy(),
    "click": np.asarray(y_val).reshape(-1),
})

# ② 对每组的 click 列使用命名聚合
# 输出列依次命名为 exposures、clicks、ctr
group_stats = val_group_df.groupby("has_unseen").agg(
    exposures=("click","size"),
    clicks=("click","sum"),
    ctr=("click","mean")
)

# ③ 用断言核对两组曝光数之和为 15,000，点击数之和为 403
assert group_stats["exposures"].sum() == 15000
assert group_stats["clicks"].sum() == 403

print(group_stats.to_string(float_format=lambda x: f"{x:.6f}"))

            exposures  clicks      ctr
has_unseen                            
False           10051     228 0.022684
True             4949     175 0.035361


In [68]:
labels = y_val.to_numpy()
unseen = has_unseen.to_numpy()
predictions = {"A": val_prob_a, "B": val_prob_b, "C": val_prob_c}
rows = []

for flag, group_name in [(False, "无未见类别"), (True, "含未见类别")]:
    mask = unseen == flag
    group_y = labels[mask]
    print(group_name, "曝光数", len(group_y), "点击数", group_y.sum())

    for model_name, prob in predictions.items():
        group_prob = prob[mask]
        auc = roc_auc_score(group_y, group_prob)
        loss = log_loss(group_y, group_prob)
        rows.append([group_name, model_name, len(group_y), auc, loss])

group_metrics = pd.DataFrame(rows, columns=["分组", "模型", "曝光数", "AUC", "Log Loss"])
print(group_metrics.to_string(index=False, float_format=lambda x: f"{x:.6f}"))

for model_name, prob in predictions.items():
    part = group_metrics[group_metrics["模型"] == model_name]
    weighted_loss = np.average(part["Log Loss"], weights=part["曝光数"])
    overall_loss = log_loss(labels, prob)
    print(f"{model_name}：分组加权 {weighted_loss:.6f}，总体 {overall_loss:.6f}")
    assert np.isclose(weighted_loss, overall_loss, rtol=0, atol=1e-7)

无未见类别 曝光数 10051 点击数 228
含未见类别 曝光数 4949 点击数 175
   分组 模型   曝光数      AUC  Log Loss
无未见类别  A 10051 0.653448  0.107492
无未见类别  B 10051 0.695976  0.106757
无未见类别  C 10051 0.695124  0.106766
含未见类别  A  4949 0.613916  0.151610
含未见类别  B  4949 0.646186  0.151197
含未见类别  C  4949 0.644908  0.151176
A：分组加权 0.122048，总体 0.122048
B：分组加权 0.121419，总体 0.121419
C：分组加权 0.121419，总体 0.121419


In [69]:
selected_model = "model_b"
selected_epoch = 5
selection_reason = (
    "On the validation set, B had a 0.000931 higher AUC than C, "
    "while their log losses differed by only about 0.000006. "
    "B also had higher AUC in both subgroups, so I chose the simpler model B."
)

## 锁定 B 后的测试评估

In [70]:
# ① 用训练时的数值预处理器处理测试集的 13 列
X_test_scaled = num_preprocessor.transform(test_df[num_cols])

# ② 用训练时每个字段自己的词表编码测试集
test_cat_ids = pd.DataFrame(index=test_df.index)
for col in cat_cols:
    test_cat_ids[col] = encode_column(test_df[col],col)

assert X_test_scaled.shape == (len(test_df), 13)
assert test_cat_ids.shape == (len(test_df), 26)
assert test_cat_ids.index.equals(test_df.index)
assert np.isfinite(X_test_scaled).all()
assert not test_cat_ids.isna().any().any()

print("测试集数值输入形状", X_test_scaled.shape)
print("测试集类别输入形状", test_cat_ids.shape)

测试集数值输入形状 (15000, 13)
测试集类别输入形状 (15000, 26)


In [71]:
from sklearn.metrics import roc_auc_score, log_loss

y_test = test_df["label"].to_numpy()
X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32)
X_cat_test = torch.tensor(test_cat_ids[cat_cols].to_numpy(), dtype=torch.long)

model_a.eval()
model_b.eval()
model_c.eval()

with torch.no_grad():
    logits_a = model_a(X_test_t)  # ① 模型 A
    logits_b = model_b(X_test_t,X_cat_test)  # ② 模型 B
    logits_c = model_c(X_test_t,X_cat_test)  # ③ 模型 C

def to_prob(logits):
    return torch.sigmoid(logits).reshape(-1).cpu().numpy()

test_predictions = {
    "常数基线": np.full(len(y_test), p_train),
    "A": to_prob(logits_a),
    "B": to_prob(logits_b),
    "C": to_prob(logits_c),
}

rows = []
for name, prob in test_predictions.items():
    assert prob.shape == y_test.shape
    auc = roc_auc_score(y_test,prob)   # ④ 测试集 AUC
    loss = log_loss(y_test,prob)  # ⑤ 测试集 Log Loss
    rows.append([name, auc, loss])

test_metrics = pd.DataFrame(rows, columns=["模型", "AUC", "Log Loss"]).set_index("模型")
print(test_metrics.to_string(float_format=lambda x: f"{x:.6f}"))

# 直接用完整精度求差，最后再格式化显示
for newer, older in [("B", "A"), ("C", "B")]:
    delta = test_metrics.loc[newer] - test_metrics.loc[older]
    print(f"{newer}−{older}：ΔAUC={delta['AUC']:+.6f}，"
          f"ΔLog Loss={delta['Log Loss']:+.6f}")

          AUC  Log Loss
模型                     
常数基线 0.500000  0.123436
A    0.655421  0.121715
B    0.690224  0.121110
C    0.689642  0.121111
B−A：ΔAUC=+0.034803，ΔLog Loss=-0.000605
C−B：ΔAUC=-0.000582，ΔLog Loss=+0.000001


In [72]:
val_predictions = {
    "常数基线": baseline_pred,
    "A": val_prob_a,
    "B": val_prob_b,
    "C": val_prob_c,
}

rows = []
for name, val_prob in val_predictions.items():
    test_prob = test_predictions[name]
    rows.append({
        "模型": name,
        "验证 AUC": roc_auc_score(y_val, val_prob),
        "验证 Log Loss": log_loss(y_val,val_prob) , # ①
        "测试 AUC": roc_auc_score(y_test,test_prob),       # ②
        "测试 Log Loss": log_loss(y_test, test_prob),
    })

experiment_results = pd.DataFrame(rows)
print(experiment_results.to_string(index=False, float_format=lambda x: f"{x:.6f}"))
summary_test = experiment_results.set_index("模型")[["测试 AUC", "测试 Log Loss"]]
np.testing.assert_array_equal(summary_test.index, test_metrics.index)
np.testing.assert_allclose(summary_test.to_numpy(), test_metrics.to_numpy(), rtol=0, atol=0)
experiment_results.to_csv("experiment_results.csv", index=False)

  模型   验证 AUC  验证 Log Loss   测试 AUC  测试 Log Loss
常数基线 0.500000     0.123676 0.500000     0.123436
   A 0.640600     0.122048 0.655421     0.121715
   B 0.681304     0.121419 0.690224     0.121110
   C 0.680239     0.121419 0.689642     0.121111


## 实验结论

在 13 个数值特征的基础上，模型 B 为 26 个类别字段加入独立的线性权重。相较仅使用数值特征的模型 A，B 的测试 AUC 提高 0.034804，Log Loss 降低 0.000605。


模型 B 的验证 AUC 比模型 C 高 0.000931，两者的验证 Log Loss 几乎持平。据此锁定第 5 轮的模型 B，随后在留出的测试集上得到 AUC 0.690224，Log Loss 0.121110。


在本次 10 万条样本、固定划分和训练配置下，加入 FM 二阶交互后，验证和测试 AUC 分别下降 0.000931 和 0.000542，两个集合上的 Log Loss 均几乎持平。本次实验未观察到 FM 相对线性模型 B 的稳定额外收益。

数据仅覆盖 day_0 开头的连续片段，使用片段内部随机分层划分。泛化到后续日期的表现需要单独验证。